# Keltner Band Trend Break on SPY
## Strategy Brief
This strategy utilizes the Keltner Channel, a volatility-based envelope set above and below an exponential moving average (EMA). The signal is generated when the SPY price breaks above the upper band, indicating a potential upward trend, or below the lower band, indicating a potential downward trend. The trade logic involves entering a long position when the price breaks above the upper band and exiting the position when it crosses back below the middle line. The strategy aims to capture trends and can outperform buy-and-hold during trending markets.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

## Phase 1 - Trading Context
In this phase, we define the necessary parameters for our strategy, including the lookback period for the EMA and the multiplier for the Average True Range (ATR) to set the Keltner Channel bands.

In [ ]:
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
EMA_PERIOD = 20
ATR_MULTIPLIER = 2.0

## Phase 2 - Data Exploration
We will download historical SPY data from Yahoo Finance, calculate the Keltner Channel indicators, and plot them over the price data.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Download SPY data
data = yf.download('SPY', start=START_DATE, end=END_DATE)

# Calculate EMA and ATR
ema = data['Close'].ewm(span=EMA_PERIOD, adjust=False).mean()
atr = data['High'].sub(data['Low']).rolling(window=EMA_PERIOD).mean()

# Calculate Keltner Channel bands
upper_band = ema + (ATR_MULTIPLIER * atr)
lower_band = ema - (ATR_MULTIPLIER * atr)

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close', color='black')
plt.plot(ema, label='EMA', color='blue')
plt.plot(upper_band, label='Upper Band', color='red')
plt.plot(lower_band, label='Lower Band', color='green')
plt.title('SPY with Keltner Channel')
plt.legend()
plt.show()

## Phase 3 - Strategy Engineering
We will define the logic for generating buy and sell signals based on the Keltner Channel and create a series to represent our position in the market.

In [ ]:
# Generate signals
signal = pd.Series(index=data.index, data=0)
signal[data['Close'] > upper_band] = 1
signal[data['Close'] < lower_band] = -1

# Define positions
positions = signal.shift(1).fillna(0)

## Phase 4 - Coding & Backtesting
We will simulate trading by shifting the positions, calculating daily returns, and plotting the equity curve.

In [ ]:
# Calculate daily returns
returns = data['Close'].pct_change().fillna(0)

# Calculate strategy returns
strategy_returns = positions * returns

# Calculate equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve', color='blue')
plt.title('Equity Curve')
plt.legend()
plt.show()

## Phase 5 - Performance Evaluation
We will evaluate the strategy using key performance metrics such as CAGR, Sharpe Ratio, Sortino Ratio, Calmar Ratio, and Maximum Drawdown, and compare it to a buy-and-hold strategy.

In [ ]:
def calculate_performance_metrics(returns):
    cagr = (equity_curve.iloc[-1] ** (252.0/len(returns))) - 1
    sharpe_ratio = np.mean(returns) / np.std(returns) * np.sqrt(252)
    downside_returns = returns[returns < 0]
    sortino_ratio = np.mean(returns) / np.std(downside_returns) * np.sqrt(252)
    max_drawdown = (equity_curve.cummax() - equity_curve).max()
    calmar_ratio = cagr / max_drawdown
    return cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown

# Strategy performance
strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_drawdown = calculate_performance_metrics(strategy_returns)

# Buy-and-hold performance
buy_and_hold_returns = returns
buy_and_hold_equity_curve = (1 + buy_and_hold_returns).cumprod()
buy_and_hold_cagr, buy_and_hold_sharpe, buy_and_hold_sortino, buy_and_hold_calmar, buy_and_hold_max_drawdown = calculate_performance_metrics(buy_and_hold_returns)

# Comparison table
performance_df = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [strategy_cagr, strategy_sharpe, strategy_sortino, strategy_calmar, strategy_max_drawdown],
    'Buy & Hold': [buy_and_hold_cagr, buy_and_hold_sharpe, buy_and_hold_sortino, buy_and_hold_calmar, buy_and_hold_max_drawdown]
})
print(performance_df)

## Phase 6 - Deploy & Monitor
We will create a function to download the latest 60 days of SPY data, compute the current signal, and print the position for today.

In [ ]:
def get_current_signal():
    recent_data = yf.download('SPY', period='60d')
    recent_ema = recent_data['Close'].ewm(span=EMA_PERIOD, adjust=False).mean()
    recent_atr = recent_data['High'].sub(recent_data['Low']).rolling(window=EMA_PERIOD).mean()
    recent_upper_band = recent_ema + (ATR_MULTIPLIER * recent_atr)
    recent_lower_band = recent_ema - (ATR_MULTIPLIER * recent_atr)
    
    if recent_data['Close'].iloc[-1] > recent_upper_band.iloc[-1]:
        print('Current Position: Long')
    elif recent_data['Close'].iloc[-1] < recent_lower_band.iloc[-1]:
        print('Current Position: Short')
    else:
        print('Current Position: Neutral')

get_current_signal()